In [1]:
import os
import pandas as pd
import numpy as np

consumers = pd.read_parquet("/uss/hdsi-prismdata/q2-ucsd-consDF.pqt")
consumers.head()

,prism_consumer_id,evaluation_date,credit_score,DQ_TARGET
0,0,2021-09-01,726.0,0.0
1,1,2021-07-01,626.0,0.0
2,2,2021-05-01,680.0,0.0
3,3,2021-03-01,734.0,0.0
4,4,2021-10-01,676.0,0.0


In [2]:
accounts = pd.read_parquet("/uss/hdsi-prismdata/q2-ucsd-acctDF.pqt")
accounts.head()

,prism_consumer_id,prism_account_id,account_type,balance_date,balance
0,3023,0,SAVINGS,2021-08-31,90.57
1,3023,1,CHECKING,2021-08-31,225.95
2,4416,2,SAVINGS,2022-03-31,15157.17
3,4416,3,CHECKING,2022-03-31,66.42
4,4227,4,CHECKING,2021-07-31,7042.90


In [3]:
transactions = pd.read_parquet("/uss/hdsi-prismdata/q2-ucsd-trxnDF.pqt")
transactions.head()

,prism_consumer_id,prism_transaction_id,category,amount,credit_or_debit,posted_date
0,3023,0,4,0.05,CREDIT,2021-04-16
1,3023,1,12,481.56,CREDIT,2021-04-30
2,3023,2,4,0.05,CREDIT,2021-05-16
3,3023,3,4,0.07,CREDIT,2021-06-16
4,3023,4,4,0.06,CREDIT,2021-07-16


In [4]:
category_mapping = pd.read_csv("/uss/hdsi-prismdata/q2-ucsd-cat-map.csv")
category_mapping

,category_id,category
0,0,SELF_TRANSFER
1,1,EXTERNAL_TRANSFER
2,2,DEPOSIT
3,3,PAYCHECK
4,4,MISCELLANEOUS
5,5,PAYCHECK_PLACEHOLDER
6,6,REFUND
7,7,INVESTMENT_INCOME
8,8,OTHER_BENEFITS
9,9,UNEMPLOYMENT_BENEFITS


In [5]:
accounts['account_type'].unique()

array(['SAVINGS', 'CHECKING', 'CREDIT CARD', 'LINE OF CREDIT',
       'MONEYMARKET', 'LOAN', 'MONEY MARKET', 'ROTH', 'MORTGAGE',
       'RETIREMENT', 'PREPAID', 'BROKERAGE', 'CONSUMER', 'CD', 'IRA',
       'AUTO', 'STUDENT', 'HSA', 'CASH MANAGEMENT', 'OTHER', '401K',
       'STOCK PLAN', 'OVERDRAFT', 'HOME EQUITY'], dtype=object)

# Feature based on Cash Flow 

In [6]:
transactions['posted_date'].dtype

dtype('O')

In [7]:
transactions['posted_date'] = pd.to_datetime(transactions['posted_date'], errors='coerce')
transactions['month_period'] = transactions['posted_date'].dt.to_period('M')
print(transactions['posted_date'].dtype)

datetime64[ns]


In [8]:
# DEPOSIT, PAYCHECK, REFUND, INVESTMENT_INCOME, OTHER_BENEFITS, UNEMPLOYMENT_BENEFITS
income_categories = [2, 3, 6, 7, 8, 9]

monthly_income = transactions[transactions['category'].isin(income_categories)].groupby('prism_consumer_id')['amount'].sum().reset_index(name='total_income')

monthly_spending = transactions[transactions['credit_or_debit'] == 'debit'].groupby('prism_consumer_id')['amount'].sum().reset_index(name='total_spending')

transactions['posted_date'] = pd.to_datetime(transactions['posted_date'], errors='coerce')
months_active = transactions.groupby('prism_consumer_id')['posted_date'].apply(
    lambda x: x.dt.to_period('M').nunique()
).reset_index(name='month_count')

df_features = consumers[['prism_consumer_id']].merge(monthly_income, on='prism_consumer_id', how='left')
df_features = df_features.merge(monthly_spending, on='prism_consumer_id', how='left')
df_features = df_features.merge(months_active, on='prism_consumer_id', how='left')

df_features = df_features.fillna(0)

df_features['monthly_net_cash_flow'] = (
    (df_features['total_income'] - df_features['total_spending']) / df_features['month_count']
).replace([np.inf, -np.inf], 0).fillna(0)

df_features[['prism_consumer_id', 'monthly_net_cash_flow']].head()

,prism_consumer_id,monthly_net_cash_flow
0,0,1334.360000
1,1,1916.287143
2,2,165.211429
3,3,1545.098571
4,4,1741.562857


In [9]:
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp
from sklearn.linear_model import LogisticRegression

df_eval = df_features.merge(consumers[['prism_consumer_id', 'DQ_TARGET']], on='prism_consumer_id')
df_eval = df_eval.dropna(subset=['DQ_TARGET'])

X = df_eval[['monthly_net_cash_flow']]
y = df_eval['DQ_TARGET']

clf = LogisticRegression()
clf.fit(X, y)

y_pred_proba = clf.predict_proba(X)[:, 1]

# Area Under Curve
auc_score = roc_auc_score(y, y_pred_proba)
print(f"AUC Score: {auc_score:.4f}")

probs_bad = y_pred_proba[y == 1]
probs_good = y_pred_proba[y == 0]

# Kolmogorov-Smirnov
ks_statistic, p_value = ks_2samp(probs_bad, probs_good)
print(f"KS Statistic: {ks_statistic:.4f}")

AUC Score: 0.5398
KS Statistic: 0.0835


# Feature based on overdraft 

In [10]:
category_mapping[category_mapping['category']=='OVERDRAFT']

,category_id,category
25,25,OVERDRAFT


In [11]:
overdraft_txns = transactions[transactions['category']==25]

In [12]:
# 2. Count the number of overdrafts per consumer
# .size() counts the rows for each consumer
overdraft_counts = overdraft_txns.groupby('prism_consumer_id').size().reset_index(name='overdraft_count')

# 3. Merge this count back to your main 'consumers' dataframe
# Use how='left' to keep all consumers, even those who didn't appear in the overdraft list
df_features = consumers.merge(overdraft_counts, on='prism_consumer_id', how='left')

# 4. Fill missing values with 0
# If a consumer wasn't in the overdraft list, the merge creates a NaN. We convert that to 0.
df_features['overdraft_count'] = df_features['overdraft_count'].fillna(0).astype(int)

# View the result
df_features[['prism_consumer_id', 'overdraft_count']]

,prism_consumer_id,overdraft_count
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0
...,...,...
14995,14995,0
14996,14996,0
14997,14997,0
14998,14998,5


In [13]:
df_features[df_features['overdraft_count']>0]

,prism_consumer_id,evaluation_date,credit_score,DQ_TARGET,overdraft_count
151,151,2021-07-01,699.0,0.0,1
376,376,2022-04-01,690.0,0.0,3
1029,1029,2021-01-01,696.0,0.0,2
1455,1455,2022-02-01,696.0,0.0,6
1532,1532,2021-04-01,749.0,0.0,1
...,...,...,...,...,...
14983,14983,2022-01-19,675.0,NaN,39
14988,14988,2022-01-20,653.0,NaN,4
14991,14991,2022-01-26,667.0,NaN,1
14992,14992,2022-03-14,709.0,NaN,10
